## Import libraries

In [ ]:
pip install sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 3.8 MB/s eta 0:00:00


In [ ]:
#from bs4 import BeautifulSoup
#import urllib.request
import string
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import itertools
import pandas as pd
import pickle
from IPython.display import display, clear_output

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
paper_x = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/kompas/hasil_scraping_kompas.xlsx')

#paper_x = pd.DataFrame(paper_x)

paper_x = paper_x.tail(564)

paper = paper_x.values.tolist()



print(paper)

[['https://nasional.kompas.com/read/2026/03/04/15384751/bupati-pekalongan-fadia-arafiq-dan-anaknya-intervensi-kadis-untuk-pakai-jasa', 'Bupati Pekalongan Fadia Arafiq dan Anaknya Intervensi Kadis untuk Pakai Jasa Perusahaan Mereka', 'Kompas.com, 4 Maret 2026, 15:38 WIB', 'JAKARTA, KOMPAS.com - Komisi Pemberantasan Korupsi (KPK) mengatakan, Bupati Pekalongan Fadia Arafiq melalui anaknya yang menjabat sebagai Anggota DPRD, Muhammad Sabiq Ashraff, dan orang kepercayaannya diduga melakukan intervensi kepada sejumlah Perangkat Daerah Pemkab Pekalongan agar memakai perusahaan mereka yakni PT RNB (Raja Nusantara Berjaya) untuk pengadaan barang dan jasa outsourcing. Hal tersebut disampaikan Deputi Penindakan dan Eksekusi KPK Asep Guntur Rahayu saat mengumumkan status tersangka terhadap Fadia Arafiq terkait kasus pengadaan barang dan jasa outsourcing di lingkungan Pemkab Pekalongan. “Pada periode tersebut (2024), FAR (Fadia Arafiq) melalui anaknya, (Muhammad Sabiq Ashraff) dan orang kepercayaan

In [ ]:
print("Saving data to pickle/paper.pkl..")
with open('/content/drive/MyDrive/Colab Notebooks/kompas/tugas.pkl', 'wb') as f:
    pickle.dump(paper, f)
print("Success.")

Saving data to pickle/paper.pkl..
Success.


## Preprocess the documents

In [ ]:
# preprocessing

factory = StopWordRemoverFactory()
stopword = factory.create_stop_word_remover()
stemmer = StemmerFactory().create_stemmer()
words = []
processed_paper = []
for x in tqdm(paper, desc='paper', unit='paper'):
    text = x[3]
    text = text.lower()

    remove_punctuation_map = dict((ord(char), None) for char in string.punctuation)
    text = text.translate(remove_punctuation_map)
    text = stopword.remove(text)
    text = text.split()
    text = [stemmer.stem(x) for x in text]
    processed_paper.append(' '.join(text))
    text = list(set(text))
    words += text

   # print (text)

paper: 100%|██████████| 49/49 [02:49<00:00,  3.47s/paper]


In [ ]:
# save results to 'corpus/processed_paper.xlsx'

print("Saving data to corpus/tugas.xlsx..")
df = pd.DataFrame(processed_paper)
df.to_excel('/content/drive/MyDrive/Colab Notebooks/kompas/processed_tugas.xlsx', header=False, index=False)
print("Success.")

Saving data to corpus/processed_quran_juz30.xlsx..
Success.


In [ ]:
# save results to 'pickle/processed_paper.pkl'

print("Saving data to pickle/processed_paper.pkl..")
with open('/content/drive/MyDrive/Colab Notebooks/kompas/processed_tgs.pkl', 'wb') as f:
    pickle.dump(processed_paper, f)
print("Success.")

Saving data to pickle/processed_paper.pkl..
Success.


In [ ]:
# save words to 'corpus/words.xlsx'

print("Saving data to corpus/lagi.xlsx..")
df = pd.DataFrame(words)
df.to_excel('/content/drive/MyDrive/Colab Notebooks/kompas/lagi.xlsx', header=False, index=False)
print("Success.")

Saving data to corpus/juz30.xlsx..
Success.


In [ ]:
# save words to 'pickle/words.pkl'

print("Saving data to pickle/hadeh.pkl..")
with open('/content/drive/MyDrive/Colab Notebooks/kompas/hadeh.pkl', 'wb') as f:
    pickle.dump(words, f)
print("Success.")

Saving data to pickle/juz30.pkl..
Success.


## Testing

### Test 1. Query: 'pengembangan aplikasi'

In [ ]:
# insert query here

init_query = 'menteri'

#### Without query expansion:

In [ ]:
# build tf_idf

vectorizer = TfidfVectorizer(use_idf=True)
query = init_query
query = query.lower()
remove_punctuation_map = dict((ord(char), None) for char in string.punctuation)
query = query.translate(remove_punctuation_map)
query = stopword.remove(query)
query = query.split()
query = [stemmer.stem(x) for x in query]
print("Query used: " +' '.join(query))

Query used: menteri


In [ ]:
# process the query

max_result = []
x = [' '.join(query)]
paper_tfidf = vectorizer.fit_transform(x + processed_paper)
q = paper_tfidf[0]
result = cosine_similarity(paper_tfidf, q)
idx = np.argsort(-result,axis=0).flatten()
final = [[num, y[0], x] for num, y in enumerate(result) if y[0] > 0.0]
max_result += final
max_result = sorted(max_result, key=lambda x: x[1], reverse=True)
set_result = set()
new_result = []
for item in max_result:
    if item[0] not in set_result:
        set_result.add(item[0])
        new_result.append(item)
    else:
        pass
print("Number of documents returned: " +str(len(new_result)-1)+ ".")
print("Top 5 [document, scores, query]:")
for x in new_result[1:6]:
    print(x)

Number of documents returned: 23.
Top 5 [document, scores, query]:
[11, np.float64(0.09672404662810236), ['menteri']]
[44, np.float64(0.07627513651900324), ['menteri']]
[36, np.float64(0.07531490528159882), ['menteri']]
[48, np.float64(0.0721627075024016), ['menteri']]
[2, np.float64(0.06321327045913071), ['menteri']]


In [ ]:
# show top results (misal ingin 5 hasil teratas yang valid)
for x in new_result[1:100]: # Mulai dari index 1, bukan dari 0
    print('Result', x[0])
    print('QUERY/SCORE', x[2])

    print('Judul :', paper[x[0]-1][1])
    print('Waktu :', paper[x[0]-1][2])
    print('URL   :', paper[x[0]-1][0])

    print('Konten:', paper[x[0]-1][3][:200], '...\n')

Result 11
QUERY/SCORE ['menteri']
Judul : Mengapa Pemerintah Batasi Akses ke TikTok hingga Roblox untuk Anak di Bawah 16 Tahun?
Waktu : Kompas.com, 7 Maret 2026, 08:47 WIB
URL   : https://nasional.kompas.com/read/2026/03/07/08474671/mengapa-pemerintah-batasi-akses-ke-tiktok-hingga-roblox-untuk-anak-di-bawah
Konten: JAKARTA, KOMPAS.com - Pemerintah Kementerian Komunikasi dan Digital mengeluarkan kebijakan baru melalui terkait penggunaan media sosial bagi anak-anak di bawah usia 16 tahun yang mulai diterapkan pada ...

Result 44
QUERY/SCORE ['menteri']
Judul : Kemlu Tegaskan Prinsip Politik Bebas Aktif RI dalam Konflik Timur Tengah
Waktu : Kompas.com, 6 Maret 2026, 20:55 WIB
URL   : https://nasional.kompas.com/read/2026/03/06/20551971/kemlu-tegaskan-prinsip-politik-bebas-aktif-ri-dalam-konflik-timur-tengah
Konten: JAKARTA, KOMPAS.com - Kementerian Luar Negeri (Kemlu) menegaskan prinsip politik bebas aktif Indonesia dalam menyikapi konflik yang terjadi di Timur Tengah, usai Amerika Serika

In [ ]:
import os
import pandas as pd

# 1. Buat folder 'result' jika belum ada
if not os.path.exists('result'):
    os.makedirs('result')

# 2. Proses menyimpan hasil
file = []
for x in new_result[1:]:
    temp = []
    temp.append('Document: ' + str(x[0]))

    # Asumsi x[2] adalah nilai skor/query. Jika x[2] list, gunakan x[2][0] seperti kode awal Anda.
    temp.append('Score/Query: ' + str(x[2]))

    # Menyesuaikan dengan urutan kolom file hasil scraping Kompas:
    temp.append('URL: ' + str(paper[x[0]-1][0]))
    temp.append('Title: ' + str(paper[x[0]-1][1]))
    temp.append('Waktu: ' + str(paper[x[0]-1][2]))
    temp.append('Konten: ' + str(paper[x[0]-1][3]))

    file.append(temp)

print("Saving result to result/" + init_query + "_original.xlsx..")
df = pd.DataFrame(file)

# 3. Simpan file
df.to_excel('result/' + init_query + '_original.xlsx', header=False, index=False)
print("Success.")

Saving result to result/menteri_original.xlsx..
Success.
